In [34]:
from pathlib import Path
from docx import Document
import zipfile
import re
from xml.etree import ElementTree as ET
from docx.oxml.ns import qn
import json

In [35]:
def parse_docx(docx_path):
    '''Сборка одного словаря'''
    doc = Document(docx_path)
    headers = read_headers_from_zip(docx_path)

    blocks = []
    current_anchor = None

    for kind, el in iter_body_blocks(doc):
        if kind == 'paragraph':
            text = paragraph_text(el).strip()
            if not text:
                continue
            # anchor = detect_anchor(text)
            # if anchor:
            #     current_anchor = anchor
            blocks.append({
                'type': 'paragraph',
                'text': text,
                'anchor': None, #'anchor': current_anchor,
            })
        else:  # table
            rows = parse_table(el)
            blocks.append({
                'type': 'table',
                'anchor': None, #'anchor': current_anchor,
                'rows': rows,
            })

    return {
        'id': docx_path.stem,
        'headers': headers,
        'blocks': blocks,
    }

def read_headers_from_zip(docx_path):
    '''Колонтитулы через прямой разбор zip'''
    texts = []
    W_NS = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'
    with zipfile.ZipFile(docx_path) as z:
        names = [n for n in z.namelist()
                 if re.match(r'word/(header|footer)\d*\.xml$', n)]
        for name in sorted(names):
            root = ET.fromstring(z.read(name))
            # ElementTree сам декодирует &amp; -> &
            paragraphs = []
            for p in root.iter(f'{{{W_NS}}}p'):
                parts = [t.text or '' for t in p.iter(f'{{{W_NS}}}t')]
                line = ''.join(parts).strip()
                if line:
                    paragraphs.append(line)
            if paragraphs:
                texts.append({'source': name, 'lines': paragraphs})
    return texts

def iter_body_blocks(doc):
    '''Итерирует w:p и w:tbl в порядке появления в body.'''
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag == qn('w:p'):
            yield ('paragraph', child)
        elif child.tag == qn('w:tbl'):
            yield ('table', child)

def paragraph_text(p_el):
    return ''.join(t.text or '' for t in p_el.iter(qn('w:t')))

def parse_table(tbl_el):
    rows_out = []
    for r_idx, tr in enumerate(tbl_el.findall(qn('w:tr'))):
        col_idx = 0
        for tc in tr.findall(qn('w:tc')):
            info = cell_info(tc)
            # пропускаем 'продолжение' вертикального объединения
            if info['vmerge'] == 'continue':
                col_idx += info['gridspan']
                continue
            rows_out.append({
                'row': r_idx,
                'col': col_idx,
                'text': info['text'],
                'gridspan': info['gridspan'],
                'vmerge': info['vmerge'],
                'is_header': r_idx == 0,
            })
            col_idx += info['gridspan']
    return rows_out

def cell_info(tc):
    '''Возвращает текст ячейки, gridSpan, vMerge.'''
    text = ''.join(t.text or '' for t in tc.iter(qn('w:t')))
    tcPr = tc.find(qn('w:tcPr'))
    gridspan = 1
    vmerge = None
    if tcPr is not None:
        gs = tcPr.find(qn('w:gridSpan'))
        if gs is not None:
            gridspan = int(gs.get(qn('w:val'), '1'))
        vm = tcPr.find(qn('w:vMerge'))
        if vm is not None:
            vmerge = vm.get(qn('w:val')) or 'continue'
    return {'text': text.strip(), 'gridspan': gridspan, 'vmerge': vmerge}

In [36]:
# docx_path = Path('sample.docx')
# out_dir = Path('parsed')
# out_dir.mkdir(exist_ok=True)

# result = parse_docx(docx_path)
# out_path = out_dir / f'{docx_path.stem}.json'
# out_path.write_text(
#     json.dumps(result, ensure_ascii=False, indent=2),
#     encoding='utf-8',
# )

41512